In [ ]:
#Instalar MolVS
!pip install -q molvs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# Instalar rdkit
!pip -q install rdkit.pypi==2021.9.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 41.8 MB/s eta 0:00:00


In [ ]:
#Importar librerías necesarias
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem import rdMolDescriptors
from molvs.standardize import Standardizer
from molvs.charge import Uncharger, Reionizer
from molvs.fragment import LargestFragmentChooser
from molvs.tautomer import TautomerCanonicalizer
from rdkit.Chem.rdmolops import GetFormalCharge, RemoveStereochemistry

In [ ]:
# Cargar los DataFrames desde archivos CSV
Leadlikeness_compounds = pd.read_csv('/content/drive/MyDrive/Doctorado/Set de Datos 2/Objetivo 4/6_Compuestos a probar/Leadlikeness_compounds.csv')
#FDA
url_fda = "https://raw.githubusercontent.com/DIFACQUIM/Cursos/main/Datasets/FDA_2022_july_05_curada.csv"
FDA = pd.read_csv(url_fda)

In [ ]:
FDA

,ID,SMILES,NEW_SMILES,Data set
0,DB00006,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,CCC(C)C(NC(=O)C(CCC(=O)O)NC(=O)C(CCC(=O)O)NC(=...,FDA
1,DB00007,CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(N)=N)NC(=...,CCNC(=O)C1CCCN1C(=O)C(CCCN=C(N)N)NC(=O)C(CC(C)...,FDA
2,DB00014,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,CC(C)CC(NC(=O)C(COC(C)(C)C)NC(=O)C(Cc1ccc(O)cc...,FDA
3,DB00027,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,CC(C)CC(NC(=O)CNC(=O)C(NC=O)C(C)C)C(=O)NC(C)C(...,FDA
4,DB00035,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,N=C(N)NCCCC(NC(=O)C1CCCN1C(=O)C1CSSCCC(=O)NC(C...,FDA
...,...,...,...,...
2304,DB16627,CCOC(=O)[C@H](CC1=CC=C(F)C=C1)NC(=O)[C@@H](N)C...,CCOC(=O)C(Cc1ccc(F)cc1)NC(=O)C(N)Cc1ccc(N(CCCl...,FDA
2305,DB16628,[H][C@@]12COP(O)(=O)O[C@]1([H])C(O)(O)[C@]1([H...,N=c1[nH]c2c(c(=O)[nH]1)NC1C(N2)OC2CO[PH](=O)(=...,FDA
2306,DB16629,[H][C@@]1(CCCCN1C(=O)OC[N+]1=CC=CC(=C1)C(=O)N[...,COC(=O)C(c1ccccc1)C1CCCCN1C(=O)OC[n+]1cccc(C(=...,FDA
2307,DB16703,CC(C)NC(=O)COC1=CC=CC(=C1)C1=NC2=C(C=CC=C2)C(N...,CC(C)NC(=O)COc1cccc(-c2nc(=Nc3ccc4[nH]ncc4c3)c...,FDA


In [ ]:
Leadlikeness_compounds

,Name,SMILES,Total Molweight,cLogP,H-Acceptors,H-Donors,Total Surface Area,Relative PSA,Rotatable Bonds,Senotherapeutics:,Leadlikeness
0,"(5-(2,4-bis((3S)-3-methylmorpholin-4-yl)pyrido...",CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=C(C=C4)O...,465.552,2.4865,9,1,351.95,0.23614,5,1,True
1,(E)-4-((2-N-(4-methoxybenzenesulfonyl)amino)st...,COC1=CC=C(C=C1)S(=O)(=O)N=C2C=CC=CC2=CC=C3C=CN...,382.439,1.7590,6,1,292.14,0.22845,3,1,True
2,10-decarbamoylmitomycin C,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N,291.306,-1.6656,7,3,197.58,0.45359,2,1,True
3,10-hydroxycamptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=C(C3=C2)N=C5C=CC(=...,364.356,0.8381,7,2,246.20,0.31194,1,1,True
4,103D5R,CC(C1=C(C2=C(C=C1)OC(C=C2)(C)C)OC)N3C=NC4=C3C=...,335.406,4.0763,5,0,259.29,0.18824,3,1,True
...,...,...,...,...,...,...,...,...,...,...,...
265,vistusertib,CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=CC=C4)C(...,462.552,2.5926,9,1,351.05,0.24073,4,1,True
266,withaferin A,CC1=C(C(=O)OC(C1)C(C)C2CCC3C2(CCC4C3CC5C6(C4(C...,470.604,2.4938,6,2,334.36,0.23905,3,1,True
267,withanone,CC1=C(C(=O)OC(C1)C(C)C2(CCC3C2(CCC4C3C5C(O5)C6...,470.604,2.5971,6,2,330.77,0.24165,2,1,True
268,zomepirac glucuronide,CC1=C(N(C(=C1)CC(=O)OC2C(C(C(C(O2)C(=O)O)O)O)O...,467.857,-0.0007,10,4,323.57,0.36589,7,1,True


In [ ]:
#Función para curado
def pretreatment(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol == None:
            #If rdkit could not parse the smiles, returns Error 1
            return "Error 1"
        else:
            mol = STD(mol)
            mol = LFC(mol)

            allowed_elements = {"H","B","C","N","O","F","Si","P","S","Cl","Se","Br","I"}
            actual_elements = set([atom.GetSymbol() for atom in mol.GetAtoms()])
            if len(actual_elements-allowed_elements) == 0:
                mol = UC(mol)
                mol = RI(mol)
                #RemoveStereochemistry(mol)
                mol = TC(mol)
                return Chem.MolToSmiles(mol)
            else:
                # If molecule contains other than the allowed elements, return "Error 2"
                return "Error 2"
    except:
        return "Something else was found"

In [ ]:
#Definir funciones
STD = Standardizer() # Get the standardized version of a given SMILES string (canonical SMILES).
LFC = LargestFragmentChooser() # Select the largest fragment from a salt (ionic compound).
UC = Uncharger() # Charge corrections are applied to ensure, for example, that free metals are correctly ionized.
RI = Reionizer() # Neutralize molecule by adding/removing hydrogens.
TC = TautomerCanonicalizer()  # Return a tautormer “reasonable” from a chemist’s point, but isn’t guaranteed to be the most energetically favourable.

In [ ]:
#Nueva columna de SMILES
Leadlikeness_compounds["NEW_SMILES"] = [pretreatment(x) for x in Leadlikeness_compounds["SMILES"]]
Leadlikeness_compounds.head()

,Name,SMILES,Total Molweight,cLogP,H-Acceptors,H-Donors,Total Surface Area,Relative PSA,Rotatable Bonds,Senotherapeutics:,Leadlikeness,NEW_SMILES
0,"(5-(2,4-bis((3S)-3-methylmorpholin-4-yl)pyrido...",CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=C(C=C4)O...,465.552,2.4865,9,1,351.95,0.23614,5,1,True,COc1ccc(-c2ccc3c(N4CCOCC4C)nc(N4CCOCC4C)nc3n2)...
1,(E)-4-((2-N-(4-methoxybenzenesulfonyl)amino)st...,COC1=CC=C(C=C1)S(=O)(=O)N=C2C=CC=CC2=CC=C3C=CN...,382.439,1.7590,6,1,292.14,0.22845,3,1,True,COc1ccc(S(=O)(=O)N=C2C=CC=CC2=CC=C2C=CN(O)C=C2...
2,10-decarbamoylmitomycin C,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N,291.306,-1.6656,7,3,197.58,0.45359,2,1,True,COC12C(C=O)c3c(O)c(N)c(C)c(O)c3N1CC1NC12
3,10-hydroxycamptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=C(C3=C2)N=C5C=CC(=...,364.356,0.8381,7,2,246.20,0.31194,1,1,True,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3cc(O)ccc3nc2-1
4,103D5R,CC(C1=C(C2=C(C=C1)OC(C=C2)(C)C)OC)N3C=NC4=C3C=...,335.406,4.0763,5,0,259.29,0.18824,3,1,True,COc1c(C(C)n2cnc3ncccc32)ccc2c1C=CC(C)(C)O2


In [ ]:
Leadlikeness_compounds

,Name,SMILES,Total Molweight,cLogP,H-Acceptors,H-Donors,Total Surface Area,Relative PSA,Rotatable Bonds,Senotherapeutics:,Leadlikeness,NEW_SMILES
0,"(5-(2,4-bis((3S)-3-methylmorpholin-4-yl)pyrido...",CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=C(C=C4)O...,465.552,2.4865,9,1,351.95,0.23614,5,1,True,COc1ccc(-c2ccc3c(N4CCOCC4C)nc(N4CCOCC4C)nc3n2)...
1,(E)-4-((2-N-(4-methoxybenzenesulfonyl)amino)st...,COC1=CC=C(C=C1)S(=O)(=O)N=C2C=CC=CC2=CC=C3C=CN...,382.439,1.7590,6,1,292.14,0.22845,3,1,True,COc1ccc(S(=O)(=O)N=C2C=CC=CC2=CC=C2C=CN(O)C=C2...
2,10-decarbamoylmitomycin C,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2CO)OC)N4)N,291.306,-1.6656,7,3,197.58,0.45359,2,1,True,COC12C(C=O)c3c(O)c(N)c(C)c(O)c3N1CC1NC12
3,10-hydroxycamptothecin,CCC1(C2=C(COC1=O)C(=O)N3CC4=C(C3=C2)N=C5C=CC(=...,364.356,0.8381,7,2,246.20,0.31194,1,1,True,CCC1(O)C(=O)OCc2c1cc1n(c2=O)Cc2cc3cc(O)ccc3nc2-1
4,103D5R,CC(C1=C(C2=C(C=C1)OC(C=C2)(C)C)OC)N3C=NC4=C3C=...,335.406,4.0763,5,0,259.29,0.18824,3,1,True,COc1c(C(C)n2cnc3ncccc32)ccc2c1C=CC(C)(C)O2
...,...,...,...,...,...,...,...,...,...,...,...,...
265,vistusertib,CC1COCCN1C2=NC(=NC3=C2C=CC(=N3)C4=CC(=CC=C4)C(...,462.552,2.5926,9,1,351.05,0.24073,4,1,True,CNC(=O)c1cccc(-c2ccc3c(N4CCOCC4C)nc(N4CCOCC4C)...
266,withaferin A,CC1=C(C(=O)OC(C1)C(C)C2CCC3C2(CCC4C3CC5C6(C4(C...,470.604,2.4938,6,2,334.36,0.23905,3,1,True,CC1=C(CO)C(=O)OC(C(C)C2CCC3C4CC5OC56C(=O)CCC(=...
267,withanone,CC1=C(C(=O)OC(C1)C(C)C2(CCC3C2(CCC4C3C5C(O5)C6...,470.604,2.5971,6,2,330.77,0.24165,2,1,True,CC1=C(C)C(=O)OC(C(C)C2(O)CCC3C4C5OC5C5(O)C=CCC...
268,zomepirac glucuronide,CC1=C(N(C(=C1)CC(=O)OC2C(C(C(C(O2)C(=O)O)O)O)O...,467.857,-0.0007,10,4,323.57,0.36589,7,1,True,Cc1cc(CC(=O)OC2OC(C(=O)O)C(O)C(O)C2O)n(C)c1C(=...


In [ ]:
# Filtrar CTD_DesMol_curado según los valores en la columna 'Name' de Leadlikeness_compounds
resultados = FDA[FDA['NEW_SMILES'].isin(Leadlikeness_compounds['NEW_SMILES'])]

In [ ]:
resultados.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49 entries, 31 to 2167
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          49 non-null     object
 1   SMILES      49 non-null     object
 2   NEW_SMILES  49 non-null     object
 3   Data set    49 non-null     object
dtypes: object(4)
memory usage: 1.9+ KB


In [ ]:
# prompt: agragar el resto de columnas

# Agregar el resto de columnas de FDA a los resultados
resultados = pd.merge(resultados, Leadlikeness_compounds, on='NEW_SMILES', how='left')
resultados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  49 non-null     object 
 1   SMILES_x            49 non-null     object 
 2   NEW_SMILES          49 non-null     object 
 3   Data set            49 non-null     object 
 4   Name                49 non-null     object 
 5   SMILES_y            49 non-null     object 
 6   Total Molweight     49 non-null     float64
 7   cLogP               49 non-null     float64
 8   H-Acceptors         49 non-null     int64  
 9   H-Donors            49 non-null     int64  
 10  Total Surface Area  49 non-null     float64
 11  Relative PSA        49 non-null     float64
 12  Rotatable Bonds     49 non-null     int64  
 13  Senotherapeutics:   49 non-null     int64  
 14  Leadlikeness        49 non-null     bool   
dtypes: bool(1), float64(4), int64(4), object(6)
memory usage: 5

In [ ]:
resultados

,ID,SMILES_x,NEW_SMILES,Data set,Name,SMILES_y,Total Molweight,cLogP,H-Acceptors,H-Donors,Total Surface Area,Relative PSA,Rotatable Bonds,Senotherapeutics:,Leadlikeness
0,DB00140,CC1=C(C)C=C2N(C[C@H](O)[C@H](O)[C@H](O)CO)C3=N...,Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(CC(O)C(O)C(O)CO)c...,FDA,Riboflavin,CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(...,376.368,-2.0663,10,5,264.40,0.44066,5,1,True
1,DB00197,CC1=C(C)C2=C(CCC(C)(COC3=CC=C(CC4SC(=O)NC4=O)C...,Cc1c(C)c2c(c(C)c1O)CCC(C)(COc1ccc(Cc3sc(=O)[nH...,FDA,Troglitazone,CC1=C(C2=C(CCC(O2)(C)COC3=CC=C(C=C3)CC4C(=O)NC...,441.546,4.3852,6,2,323.81,0.27217,5,1,True
2,DB00222,CCC1=C(C)CN(C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)NC(=...,CCc1c(C)cn(C(=O)NCCc2ccc(S(=O)(=O)NC(=O)NC3CCC...,FDA,glimepiride,CCC1=C(CN(C1=O)C(=O)NCCC2=CC=C(C=C2)S(=O)(=O)N...,490.623,3.5186,9,3,367.55,0.28739,6,1,True
3,DB00227,[H][C@]12[C@H](C[C@@H](C)C=C1C=C[C@H](C)[C@@H]...,CCC(C)C(=O)OC1CC(C)C=C2C=CC(C)C(CCC3CC(O)CC(=O...,FDA,Lovastatin,CCC(C)C(=O)OC1CC(C=C2C1C(C(C=C2)C)CCC3CC(CC(=O...,404.545,3.8950,5,1,317.66,0.18630,7,1,True
4,DB00266,OC1=C(CC2=C(O)C3=C(OC2=O)C=CC=C3)C(=O)OC2=C1C=...,O=c1c(Cc2c(O)oc3ccccc3c2=O)c(O)oc2ccccc12,FDA,Dicumarol,C1=CC=C2C(=C1)C(=C(C(=O)O2)CC3=C(C4=CC=CC=C4OC...,336.298,2.5880,6,2,233.82,0.30913,2,1,True
5,DB00275,CCCC1=NC(=C(N1CC1=CC=C(C=C1)C1=C(C=CC=C1)C1=NN...,CCCc1nc(C(C)(C)O)c(C(=O)O)n1Cc1ccc(-c2ccccc2-c...,FDA,olmesartan,CCCC1=NC(=C(N1CC2=CC=C(C=C2)C3=CC=CC=C3C4=NNN=...,446.509,3.2081,9,3,345.27,0.30116,8,1,True
6,DB00301,[H][C@]12SC(C)(C)[C@@H](N1C(=O)[C@H]2NC(=O)C1=...,Cc1onc(-c2c(F)cccc2Cl)c1C(=O)NC1C(=O)N2C1SC(C)...,FDA,Floxacillin,CC1=C(C(=NO1)C2=C(C=CC=C2Cl)F)C(=O)NC3C4N(C3=O...,453.877,2.8046,8,2,299.78,0.36627,4,1,True
7,DB00305,[H][C@]12CN3C4=C([C@@H](COC(N)=O)[C@@]3(OC)[C@...,COC12C(=COC(N)=O)c3c(O)c(N)c(C)c(O)c3N1CC1NC12,FDA,Mitomycin,CC1=C(C(=O)C2=C(C1=O)N3CC4C(C3(C2COC(=O)N)OC)N4)N,334.331,-1.8035,9,3,227.50,0.50475,4,1,True
8,DB00334,CN1CCN(CC1)C1=NC2=CC=CC=C2NC2=C1C=C(C)S2,Cc1cc2c(s1)=Nc1ccccc1NC=2N1CCN(C)CC1,FDA,Olanzapine,CC1=CC2=C(S1)NC3=CC=CC=C3N=C2N4CCN(CC4)C,312.440,3.0388,4,1,236.55,0.21319,0,1,True
9,DB00338,COC1=CC2=C(C=C1)N=C(N2)S(=O)CC1=NC=C(C)C(OC)=C1C,COc1ccc2[nH]c([S+]([O-])Cc3ncc(C)c(OC)c3C)nc2c1,FDA,Omeprazole,CC1=CN=C(C(=C1OC)C)CS(=O)C2=NC3=C(N2)C=C(C=C3)OC,345.422,2.0920,6,1,260.23,0.30223,5,1,True


In [ ]:
# Guardar los resultados en un archivo CSV
resultados.to_csv('resultados_Lead_FDA.csv', index=False)